# 02 — Load the local BF16 checkpoint and freeze the tuning baseline

Notebook 01 used NVIDIA's hosted Lightning and Ultra NVFP4 services without a GPU. This notebook loads the pinned Lightning BF16 customization checkpoint that Notebook 03 will actually tune, inspects its 30B-total/3B-active hybrid MoE architecture, and freezes the exact local baseline used for the PEFT claim.

The target codes (`B77_00`–`B77_76`) are intentionally private and opaque. A fixed recorded permutation prevents the public category order from revealing the mapping. This simulates a real internal routing taxonomy: the base model understands banking language, but it cannot know which internal code your organization assigned to each intent. The official BANKING77 test split remains untouched by training.

Repeating the baseline is intentional: API-versus-tuned would confound LoRA with quantization and serving differences. **Expected time after model prefetch:** 10–15 minutes on one H100 80 GB.

In [ ]:
from pathlib import Path
import gc, json, os, subprocess, sys, time

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Repository:', ROOT)

## 1. Preflight and model identity

The lab uses NVIDIA's full-precision customization checkpoint, not the NVFP4 deployment checkpoint. BF16 is the correct starting point for SFT; optimized deployment quantization belongs after tuning.

In [ ]:
subprocess.run([sys.executable, 'scripts/preflight.py', '--profile', 'inference'], check=True)

from nemotron_ft_lab.constants import MODEL_ID, MODEL_REVISION
print('Model:', MODEL_ID)
print('Pinned revision:', MODEL_REVISION)

### Optional Hugging Face authentication

The public inputs normally work anonymously. If a deployment encounters a Hub rate limit, set the flag below to `True`; `getpass` keeps the token out of notebook output and the login is not written to Git credentials.

In [ ]:
AUTHENTICATE_HF = False
if AUTHENTICATE_HF:
    from getpass import getpass
    from huggingface_hub import login

    hub_token = getpass('Hugging Face token (input hidden): ')
    login(token=hub_token, add_to_git_credential=False)
    del hub_token

## 2. Build the deterministic data bundle

The default workshop bundle takes 25 train, 5 validation, and 3 test examples for each of 77 labels. Validation is removed from the official train split before the train sample is selected; evaluation comes only from the official test split. The manifest records every example ID.

In [ ]:
DATA_DIR = ROOT / 'artifacts/data/banking77'
subprocess.run([
    sys.executable, 'scripts/prepare_banking77.py',
    '--output-dir', str(DATA_DIR),
], check=True)
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
manifest['counts'], manifest['official_split_policy']

In [ ]:
from nemotron_ft_lab.data import balanced_evaluation_subset, read_jsonl

eval_rows = read_jsonl(DATA_DIR / 'evaluation.jsonl')
LOCAL_EXAMPLES_PER_LABEL = 3
local_eval_rows = balanced_evaluation_subset(
    eval_rows, examples_per_label=LOCAL_EXAMPLES_PER_LABEL,
)
label_map = json.loads((DATA_DIR / 'label_map.json').read_text())
assert set(manifest['train_example_ids']).isdisjoint(manifest['validation_example_ids'])
assert all(x.startswith('test-') for x in manifest['test_example_ids'])
print('Frozen local evaluation rows:', len(local_eval_rows))
print('Example:', {k: local_eval_rows[0][k] for k in ('utterance', 'expected', 'label_name')})
print('Private mapping example:', list(label_map.items())[:3])

## 3. Load the BF16 model

On one 80 GB GPU the weights consume roughly 60 GB, so keep the context and generation short. `device_map='auto'` permits bounded CPU offload if the runtime needs a little headroom.

In [ ]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

config = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True)
print('Architecture:', config.architectures)
print('Layers:', getattr(config, 'num_hidden_layers', 'see nested config'))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True
)
load_started = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
print(f'Loaded in {(time.perf_counter() - load_started):.1f}s')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B total')

## 4. Inspect an ordinary output, then run the exact BF16 benchmark

First ask a normal question to see that the untuned model is capable. Then give it the private routing contract. Accuracy and valid-code rate are different: emitting a well-formed but wrong code is not task success.

In [ ]:
messages = [{'role': 'user', 'content': 'Explain in two sentences why a bank transfer can remain pending.'}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, enable_thinking=False,
    tokenize=True, return_dict=True, return_tensors='pt'
).to(model.device)
with torch.inference_mode():
    output = model.generate(**inputs, do_sample=False, max_new_tokens=80)
print(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
from nemotron_ft_lab.evaluation import generate_predictions, save_report, score_predictions

started = time.perf_counter()
baseline_rows = generate_predictions(model, tokenizer, local_eval_rows, batch_size=8)
baseline = score_predictions(baseline_rows)
baseline['wall_time_seconds'] = time.perf_counter() - started
baseline['precision'] = 'BF16'
baseline['checkpoint_revision'] = MODEL_REVISION
save_report(
    ROOT / 'artifacts/evaluation/baseline_local_bf16.json', baseline,
    model=f'{MODEL_ID}@{MODEL_REVISION}', run_type='untuned-local-bf16-baseline'
)
{k: baseline[k] for k in ('n', 'accuracy', 'macro_accuracy', 'valid_code_rate', 'wall_time_seconds')}

In [ ]:
errors = [row for row in baseline['rows'] if not row['correct']]
for row in errors[:10]:
    print(f"{row['utterance']!r}\n  expected={row['expected']} ({row['label_name']}) generated={row['generated']!r}\n")

## 5. Compare serving contexts without calling it a tuning result

If either Notebook 01 report is available, compare the hosted model with pinned local Lightning BF16 on the shared IDs. The cloud smoke profile can contain 77 rows while the local report contains 231, so the code aligns by example ID rather than requiring equal report sizes. These are model/serving/precision comparisons, not fine-tuning gains.

In [ ]:
from nemotron_ft_lab.evaluation import paired_accuracy_comparison

cloud_paths = sorted((ROOT / 'artifacts/evaluation').glob(
    'baseline_api_*_nvfp4_*_per_label.json'
))
baseline_by_id = {row['example_id']: row for row in baseline['rows']}
if not cloud_paths:
    print('No API reports found. This does not block the required local BF16 baseline.')
for cloud_path in cloud_paths:
    cloud = json.loads(cloud_path.read_text())
    cloud_ids = [row['example_id'] for row in cloud['rows']]
    missing = set(cloud_ids).difference(baseline_by_id)
    if missing:
        print(f'{cloud_path.name}: evaluation IDs do not match this data bundle; skipped.')
        continue
    local_matched = score_predictions([baseline_by_id[item] for item in cloud_ids])
    serving_comparison = paired_accuracy_comparison(cloud, local_matched)
    serving_comparison.update({
        'cloud_model': cloud['model'],
        'prompt_mode': cloud.get('prompt_mode', 'unspecified'),
        'cloud_nvfp4_accuracy': cloud['accuracy'],
        'local_lightning_bf16_accuracy_on_shared_ids': local_matched['accuracy'],
        'interpretation': 'local Lightning BF16 minus hosted NVFP4; not a tuning delta',
    })
    print(f'\n{cloud_path.name}')
    print(json.dumps(serving_comparison, indent=2))

## 6. Release the GPU before PEFT

Notebook 03 needs essentially the whole GPU. Delete the baseline model now and shut down this notebook's kernel before starting PEFT if Jupyter leaves it running.

In [ ]:
del model, tokenizer, inputs, output
gc.collect()
torch.cuda.empty_cache()
free_gib, total_gib = (value / 1024**3 for value in torch.cuda.mem_get_info())
print(f'GPU memory after release: {free_gib:.1f}/{total_gib:.1f} GiB free')

## Result contract

Notebook 03 must reuse `artifacts/data/banking77/evaluation.jsonl` and compare against `artifacts/evaluation/baseline_local_bf16.json`. Do not resample after seeing results. The hosted Lightning and Ultra reports remain useful targets; the isolated fine-tuning gain still means higher exact held-out accuracy than this local BF16 baseline on identical IDs—not merely lower training loss or nicer-looking examples.